# Busca Semântica com ChromaDB

Bora lá né

In [3]:
%pip install chromadb

## Tema escolhido: Filmes

Filmes mais 2000's tipo Mean Girls, De Repente 30, etc...

In [10]:
{"id", "nomeFilme", "descricao", "genero"}

{'descricao', 'genero', 'id', 'nomeFilme'}

In [11]:
# Base com 10 filmes dos anos 2000: (id, descricao, metadados)
DOCUMENTOS = [
    (
        "filme-01",
        "Uma adolescente que cresceu na África entra pela primeira vez em uma escola pública e acaba se infiltrando no grupo das garotas mais populares e fofoqueiras do colégio.",
        {"nomeFilme": "Meninas Malvadas (Mean Girls)", "genero": "Comédia", "categoria": "teen"}
    ),
    (
        "filme-02",
        "Após passar vergonha em sua festa de 13 anos, uma garota faz um pedido de aniversário e acorda magicamente no futuro como uma editora de revista de 30 anos em Nova York.",
        {"nomeFilme": "De Repente 30", "genero": "Comédia Romântica", "categoria": "romance"}
    ),
    (
        "filme-03",
        "Uma jovem apaixonada por moda e cor-de-rosa decide entrar no curso de Direito em Harvard para reconquistar o ex-namorado, mas descobre seu próprio talento como advogada.",
        {"nomeFilme": "Legalmente Loira", "genero": "Comédia", "categoria": "teen"}
    ),
    (
        "filme-04",
        "Dois agentes do FBI atrapalhados precisam usar disfarces completos para se passarem por duas herdeiras milionárias em um fim de semana de luxo nos Hamptons.",
        {"nomeFilme": "As Branquelas", "genero": "Comédia", "categoria": "besteirol"}
    ),
    (
        "filme-05",
        "Uma jornalista recém-formada consegue emprego como assistente da temida e exigente editora-chefe da maior revista de moda de Nova York.",
        {"nomeFilme": "O Diabo Veste Prada", "genero": "Comédia Dramática", "categoria": "carreira"}
    ),
    (
        "filme-06",
        "Após comerem biscoitos da sorte em um restaurante chinês, uma mãe rígida e sua filha adolescente rebelde trocam de corpo na véspera de um casamento.",
        {"nomeFilme": "Sexta-Feira Muito Louca", "genero": "Comédia", "categoria": "teen"}
    ),
    (
        "filme-07",
        "Uma colunista escreve um artigo sobre como afastar qualquer namorado em dez dias, enquanto um publicitário faz uma aposta para fazê-la se apaixonar no mesmo prazo.",
        {"nomeFilme": "Como Perder um Homem em 10 Dias", "genero": "Comédia Romântica", "categoria": "romance"}
    ),
    (
        "filme-08",
        "Um guitarrista falido e expulso de sua banda finge ser professor substituto em um colégio tradicional e transforma os alunos em uma banda de rock.",
        {"nomeFilme": "Escola de Rock", "genero": "Comédia Musical", "categoria": "teen"}
    ),
    (
        "filme-09",
        "Na década de 1940, uma jovem rica passa as férias de verão no litoral e vive um amor intenso com um trabalhador local, apesar da oposição de sua família.",
        {"nomeFilme": "Diário de uma Paixão", "genero": "Drama Romântico", "categoria": "romance"}
    ),
    (
        "filme-10",
        "Uma estudante do ensino médio finge ser seu irmão gêmeo em um internato para jogar no time de futebol masculino e provar seu talento em campo.",
        {"nomeFilme": "Ela é o Cara", "genero": "Comédia Romântica", "categoria": "teen"}
    )
]

In [12]:
import chromadb

# Inicializa o cliente e cria (ou recupera) a collection
client = chromadb.Client()
collection = client.get_or_create_collection(name="filmes_anos_2000")

# Separa os elementos da tupla: (id, descricao, metadados)
ids = [doc[0] for doc in DOCUMENTOS]
descricoes = [doc[1] for doc in DOCUMENTOS]
metadados = [doc[2] for doc in DOCUMENTOS]

# Insere (ou atualiza) os documentos na collection
collection.upsert(
    ids=ids,
    documents=descricoes,
    metadatas=metadados
)

print(f"Total de filmes armazenados: {collection.count()}")

C:\Users\manub\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:03<00:00, 21.1MiB/s]


Total de filmes armazenados: 10


### Etapa 3: Recuperar um embedding e mostrar suas dimensões

In [13]:
# Busca o registro 'filme-01' incluindo explicitamente o vetor (embedding)
registro = collection.get(ids=["filme-01"], include=["embeddings", "documents", "metadatas"])
vetor = registro["embeddings"][0]

print(f"Filme: {registro['metadatas'][0]['nomeFilme']}")
print(f"Quantidade de dimensões do embedding: {len(vetor)}")
print(f"Primeiros 5 valores do vetor: {vetor[:5]}")

Filme: Meninas Malvadas (Mean Girls)
Quantidade de dimensões do embedding: 384
Primeiros 5 valores do vetor: [ 0.01933575  0.06926773 -0.0774468   0.00104096 -0.06194958]


### Etapas 4, 5 e 7: 3 Consultas Semânticas (Top 3 resultados)
As perguntas abaixo usam palavras diferentes das sinopses originais (atendendo à **Etapa 7**), exibindo **ID, Documento, Metadata e Distância** (**Etapa 5**).

In [14]:
PERGUNTAS = [
    # Consulta 1 (Sem usar 'jornalista', 'assistente' ou 'editora-chefe' -> O Diabo Veste Prada)
    "Moça começa a trabalhar no mundo fashion com uma chefe super exigente",
    
    # Consulta 2 (Sem usar 'FBI', 'disfarces' ou 'herdeiras milionárias' -> As Branquelas)
    "Policiais se vestem de mulheres ricas para investigar um caso",
    
    # Consulta 3 / Etapa 7 (Sem usar '13 anos', 'aniversário' ou '30 anos' -> De Repente 30)
    "Dormiu quando ainda era criança e despertou já mulher madura e independente"
]

for i, pergunta in enumerate(PERGUNTAS, start=1):
    print(f"\n{'='*70}")
    print(f"PERGUNTA {i}: '{pergunta}'")
    print(f"{'='*70}")
    
    resultados = collection.query(
        query_texts=[pergunta],
        n_results=3
    )
    
    for idx in range(len(resultados["ids"][0])):
        doc_id = resultados["ids"][0][idx]
        doc_texto = resultados["documents"][0][idx]
        doc_meta = resultados["metadatas"][0][idx]
        distancia = resultados["distances"][0][idx]
        
        print(f"\n[{idx+1}º Resultado]")
        print(f"  ID:        {doc_id}")
        print(f"  Documento: {doc_texto}")
        print(f"  Metadata:  {doc_meta}")
        print(f"  Distância: {distancia:.4f}")


PERGUNTA 1: 'Moça começa a trabalhar no mundo fashion com uma chefe super exigente'

[1º Resultado]
  ID:        filme-06
  Documento: Após comerem biscoitos da sorte em um restaurante chinês, uma mãe rígida e sua filha adolescente rebelde trocam de corpo na véspera de um casamento.
  Metadata:  {'categoria': 'teen', 'nomeFilme': 'Sexta-Feira Muito Louca', 'genero': 'Comédia'}
  Distância: 0.9347

[2º Resultado]
  ID:        filme-05
  Documento: Uma jornalista recém-formada consegue emprego como assistente da temida e exigente editora-chefe da maior revista de moda de Nova York.
  Metadata:  {'nomeFilme': 'O Diabo Veste Prada', 'categoria': 'carreira', 'genero': 'Comédia Dramática'}
  Distância: 0.9858

[3º Resultado]
  ID:        filme-10
  Documento: Uma estudante do ensino médio finge ser seu irmão gêmeo em um internato para jogar no time de futebol masculino e provar seu talento em campo.
  Metadata:  {'genero': 'Comédia Romântica', 'nomeFilme': 'Ela é o Cara', 'categoria': 'teen

### Etapa 6: Consulta filtrando por categoria com `where`

In [15]:
# Filtra apenas os filmes da categoria 'romance'
pergunta_filtrada = "Casal improvável que acaba se apaixonando"

resultado_where = collection.query(
    query_texts=[pergunta_filtrada],
    n_results=3,
    where={"categoria": "romance"}
)

print(f"Busca: '{pergunta_filtrada}' | Filtro: categoria == 'romance'\n")
for idx in range(len(resultado_where["ids"][0])):
    print(f"{idx+1}. ID: {resultado_where['ids'][0][idx]} | "
          f"Filme: {resultado_where['metadatas'][0][idx]['nomeFilme']} | "
          f"Metadata: {resultado_where['metadatas'][0][idx]} | "
          f"Distância: {resultado_where['distances'][0][idx]:.4f}")
    print(f"   Documento: {resultado_where['documents'][0][idx]}\n")

Busca: 'Casal improvável que acaba se apaixonando' | Filtro: categoria == 'romance'

1. ID: filme-07 | Filme: Como Perder um Homem em 10 Dias | Metadata: {'categoria': 'romance', 'genero': 'Comédia Romântica', 'nomeFilme': 'Como Perder um Homem em 10 Dias'} | Distância: 1.0143
   Documento: Uma colunista escreve um artigo sobre como afastar qualquer namorado em dez dias, enquanto um publicitário faz uma aposta para fazê-la se apaixonar no mesmo prazo.

2. ID: filme-02 | Filme: De Repente 30 | Metadata: {'genero': 'Comédia Romântica', 'nomeFilme': 'De Repente 30', 'categoria': 'romance'} | Distância: 1.1815
   Documento: Após passar vergonha em sua festa de 13 anos, uma garota faz um pedido de aniversário e acorda magicamente no futuro como uma editora de revista de 30 anos em Nova York.

3. ID: filme-09 | Filme: Diário de uma Paixão | Metadata: {'nomeFilme': 'Diário de uma Paixão', 'categoria': 'romance', 'genero': 'Drama Romântico'} | Distância: 1.2061
   Documento: Na década de 1940,

### Pergunta para Entregar
**Por que um banco vetorial consegue encontrar um documento relevante mesmo quando a pergunta não contém exatamente as mesmas palavras do documento armazenado?**

**Resposta:**  
Diferente de uma busca tradicional por palavras-chave (que compara texto literal), um banco vetorial utiliza um modelo de linguagem para converter cada texto em um **embedding**, que é uma representação numérica na forma de um **vetor** de várias dimensões (neste caso, 384 dimensões).  

Durante o treinamento do modelo de embedding, frases e palavras com contextos e ideias parecidas aprendem a ocupar posições próximas nesse espaço matemático multidimensional. Assim, quando fazemos uma pergunta com palavras diferentes, ela também é transformada em um vetor e o ChromaDB calcula a **similaridade semântica** (medindo a distância geométrica, como distância L2 ou cosseno, entre o vetor da pergunta e os vetores dos documentos). Quanto menor a distância entre os vetores, mais próximo é o significado real dos textos, independentemente de compartilharem os mesmos termos exatos.